<a href="https://colab.research.google.com/github/SkyShealy/GPT-Creation/blob/main/RunGPTNotebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Imports & Mounting Drive**



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
import tiktoken
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**GPU/CPU Selection**

In [ ]:
def find_device():
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")
    if torch.mps.is_available():
        print("Using Apple MPS")
        return torch.device("mps")
    print("Using CPU")
    return torch.device("cpu")

device = find_device()

GPU: Tesla T4


**Model Architecture**

In [ ]:
# ===== ROTARY POSITION EMBEDDINGS (RoPE) =====
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0
        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = freqs.repeat_interleave(2, dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    def rotate_half(x):
        x_even = x[..., 0::2]
        x_odd  = x[..., 1::2]
        return torch.stack([-x_odd, x_even], dim=-1).flatten(-2)

    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)


# ===== CAUSAL MASK =====
def create_causal_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)


# ===== RMS NORMALIZATION =====
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight


# ===== SwiGLU FEED-FORWARD =====
class SwiGLU(nn.Module):
    def __init__(self, d_model, expansion_factor=4):
        super().__init__()
        hidden_dim = expansion_factor * d_model
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


# ===== MULTI-HEAD ATTENTION =====
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rotary = RotaryPositionalEmbedding(self.head_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = self.rotary(q, seq_len)
        k = self.rotary(k, seq_len)
        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        attn_output = attn_weights @ v
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_model)
        output = self.out_proj(attn_output)
        output = self.resid_dropout(output)
        return output


# ===== TRANSFORMER BLOCK =====
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


# ===== GPT CONFIGURATION =====
@dataclass
class GPTConfig:
    vocab_size: int = 50257
    d_model: int = 256
    num_heads: int = 4
    num_layers: int = 4
    max_seq_len: int = 128
    dropout: float = 0.1
    embd_dropout: float = 0.1
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    warmup_steps: int = 50
    max_steps: int = 500
    batch_size: int = 4
    grad_accum_steps: int = 2
    betas: tuple = (0.9, 0.95)
    eps: float = 1e-8


# ===== GPT MODEL =====
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.embd_dropout = nn.Dropout(config.embd_dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(config.d_model, config.num_heads, config.dropout)
            for _ in range(config.num_layers)
        ])
        self.final_norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if hasattr(module, 'bias') and module.bias is not None:
                torch.nn.init.zeros_(module.bias)

    def forward(self, input_ids, targets=None):
        batch_size, seq_len = input_ids.shape
        x = self.token_embedding(input_ids)
        x = self.embd_dropout(x)
        mask = create_causal_mask(seq_len, input_ids.device)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            logits_flat = logits.contiguous().view(-1, self.config.vocab_size)
            targets_flat = targets.contiguous().view(-1)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
        self.eval()
        for _ in range(max_new_tokens):
            if input_ids.shape[1] > self.config.max_seq_len:
                input_ids = input_ids[:, -self.config.max_seq_len:]
            logits, _ = self.forward(input_ids)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, -1:]] = float('-inf')
            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_mask = cumulative_probs > top_p
                sorted_mask[:, 1:] = sorted_mask[:, :-1].clone()
                sorted_mask[:, 0] = False
                mask = sorted_mask.scatter(1, sorted_indices, sorted_mask)
                logits[mask] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
        return input_ids

**Tokenizer**

In [ ]:
@dataclass
class TokenizerConfig:
    name: str = "gpt2"
    vocab_size: int = 50257


class SimpleTokenizer:
    def __init__(self, config=None):
        self.config = config or TokenizerConfig()
        self.enc = tiktoken.get_encoding(self.config.name)
        self.eos_token = "<|endoftext|>"
        self.eos_token_id = self.enc.encode(
            self.eos_token, allowed_special={self.eos_token}
        )[0]

    def encode(self, text):
        return self.enc.encode(text, allowed_special={self.eos_token})

    def decode(self, ids):
        return self.enc.decode(ids)

    @property
    def vocab_size(self):
        return self.config.vocab_size

tokenizer = SimpleTokenizer()

**Plot Loss Function**

In [ ]:
def plot_loss(loss_history, val_loss_history=None, save_path="loss_curve.png"):
    steps, losses = zip(*loss_history)
    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses, label="Train")
    if val_loss_history:
        v_steps, v_losses = zip(*val_loss_history)
        plt.plot(v_steps, v_losses, label="Val")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=100)
    plt.close()
    print(f"Loss curve saved to {save_path}")

**Load & Plot Checkpoint**

In [ ]:
checkpoint = torch.load(
    # Filepath for Checkpoint goes here
    "/content/drive/MyDrive/gpt_checkpoints/checkpoint_latest.pt",
    map_location=device,
    weights_only=False
)
config = checkpoint["config"]
model = GPT(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Resumed from step {checkpoint['step']}")
print(f"Parameters: {model.get_num_params():,}")
plot_loss(checkpoint["loss_history"], checkpoint["val_loss_history"])

final_train_step, final_train_loss = checkpoint["loss_history"][-1]
final_val_step, final_val_loss = checkpoint["val_loss_history"][-1]

print(f"Final train loss: {final_train_loss:.4f} at step {final_train_step}")
print(f"Final val loss:   {final_val_loss:.4f} at step {final_val_step}")

Resumed from step 28000
Parameters: 59,294,720
Loss curve saved to loss_curve.png
Final train loss: 3.3858 at step 28000
Final val loss:   3.3797 at step 28000


**Generate Text**

In [ ]:
prompts = [
    "The history of artificial intelligence",
    "In the beginning the universe",
    "The most important scientific discovery is",
    "The greatest historical figure of all time's name is",
]

for prompt in prompts:
    input_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
    output_ids = model.generate(input_ids, max_new_tokens=50, temperature=0.8, top_k=50)
    text = tokenizer.decode(output_ids[0].tolist())
    print(f"Prompt: {prompt}")
    print(f"Output: {text}")
    print("_" * 50)
    print()

Prompt: The history of artificial intelligence
Output: The history of artificial intelligence , which also includes the ability to create a scientific basis for new scientific instruments , the need to use a different method to produce a complete research . The development of the new method , codenamed the " science of the time , " was first performed
__________________________________________________

Prompt: In the beginning the universe
Output: In the beginning the universe . 
<|endoftext|> = = = Themes = = = 
<|endoftext|> Although the stories that describe the series are similar to those of other science fiction novels , they are both about the power of the series and their own lives . 
<|endoftext|> The
__________________________________________________

Prompt: The most important scientific discovery is
Output: The most important scientific discovery is the hypothesis of the origin of the term " terrestrial " , which is given by some historians as a new synonymized name . 
<|end

**Checking Tokenization that Produced Bioregia**

In [ ]:
input_ids = torch.tensor([tokenizer.encode("The most important scientific discovery is the observation of the various organisms with different bio")], dtype=torch.long)
for tid in input_ids[0][-5:]:
    print(tid.item(), repr(tokenizer.decode([tid.item()])))

2972 ' various'
20296 ' organisms'
351 ' with'
1180 ' different'
13401 ' bio'


**Checking if Bioregia Appears in Training Data**

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train", streaming=True)

found = False
for i, item in enumerate(dataset):
    if "bioregia" in item["text"].lower():
        print(f"FOUND at document {i}:")
        print(item["text"])
        found = True
        break

if not found:
    print("Not found in training data.")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Not found in training data.
